# **肌電訊號量測實作單元(二)：給我再用力一點！我知道你在偷懶！**


> 在這次的實驗中，鐵克將會帶大家一步步嘗試用RMS、IEMG、MA來分析自己的肌電訊號唷！


# 0. 先收錄一些會用到的公式和函式吧！

In [1]:
#@title
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
from scipy import signal
import plotly.graph_objects as go

def butter_bandpass(low_cut, high_cut, fs, order=3):
    """
    Design band pass filter.

    Args:
        - low_cut  (float) : the low cutoff frequency of the filter.
        - high_cut (float) : the high cutoff frequency of the filter.
        - fs       (float) : the sampling rate.
        - order      (int) : order of the filter, by default defined to 5.
    """
    # calculate the Nyquist frequency
    nyq = 0.5 * fs

    # design filter
    low = low_cut / nyq
    high = high_cut / nyq
    b, a = butter(order, [low, high], btype='band')

    # returns the filter coefficients: numerator and denominator
    return b, a 

from scipy.signal import butter, lfilter
def butter_bandpass_filter(data, lowcut, highcut, fs, order=5):
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    y = lfilter(b, a, data)
    return y
def moving_average(x, w):
    return np.convolve(x, np.ones(w), 'valid') / w

# 1.先將自己間歇出力的肌電訊號丟進程式庫吧！

In [ ]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

In [ ]:
df = pd.read_csv('範例肌電訊號1.txt')  #記得修改('xxxx')裡面的檔名唷！
df.info()
data = np.array(df)
data2 = data[0 : , 0]

# 2. 畫出自己尚未做任何處理的肌電訊號長什麼樣子吧！

In [ ]:
#稍微設定一下畫布資訊！
start_time = 2 #單位：秒
end_time = 8 #單位：秒

fig = go.Figure()
fig.add_trace(go.Scatter(
    y=data2[start_time*1000:end_time*1000], #取用其中一段肌電訊號
    mode='lines',
    name='Original Plot',
)
)
fig.update_layout(
    title="其中一段肌電訊號",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
fig.show()

# 3. 根據論文描述，用帶通濾波器將肌電訊號的頻帶範圍(20Hz~400Hz)給擷取出來吧！

In [ ]:
data3 = butter_bandpass_filter(data2, 20, 400, 1000)

start_time = 2 #單位：秒
end_time = 8 #單位：秒

#稍微設定一下畫布資訊！
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=data3[start_time*1000:end_time*1000], #取用其中一段肌電訊號
    mode='lines',
    name='Original Plot',
)
)
fig.update_layout(
    title="其中一段肌電訊號",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
fig.show()

# 4. 將濾波完的肌電訊號透過RMS方法搭配MA方法分析看看吧！



> RMS方法需要定義範圍，把整段時間的肌電訊號一次做RMS，出來一個RMS值毫無意義。



> 取合適的一段時間間隔做RMS才能夠有效處理肌電訊號




In [6]:
package = 50 #單位：毫秒                                #要不要試著調整看看package的大小呀~~
pointer = 0
loop = np.size(data3)-package
RMS_EMG = np.zeros(loop)

for i in data3:
  raw = data3[pointer:pointer+package-1]
  RMS_EMG[pointer] = np.sqrt(np.mean(raw**2))
  pointer+=1
  if(pointer == loop):
    break



In [ ]:
#稍微設定一下畫布資訊！
start_time = 2 #單位：秒
end_time = 8 #單位：秒
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=RMS_EMG[start_time*1000:end_time*1000], #取用其中一段肌電訊號
    mode='lines',
    name='Original Plot',
)
)
fig.update_layout(
    title="其中一段的肌電訊號",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
fig.show()

In [ ]:
#將尚不平滑的RMS_EMG訊號，透過MA方法優化吧！
scale = 200 #單位：毫秒
MA_RMS_EMG = moving_average(RMS_EMG,scale)


#稍微設定一下畫布資訊！
start_time = 2 #單位：秒
end_time = 8 #單位：秒

fig = go.Figure()
fig.add_trace(go.Scatter(
    y=MA_RMS_EMG[start_time*1000:end_time*1000], #取用其中一段肌電訊號
    mode='lines',
    name='Original Plot',
)
)
fig.update_layout(
    title="其中一段的肌電訊號",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
fig.show()

# 5. 將濾波完的肌電訊號透過IEMG方法搭配MA方法分析看看吧！

> IEMG方法需要定義範圍，把整段時間的肌電訊號一次做IEMG，出來一個IEMG值毫無意義。



> 取合適的一段時間間隔做IEMG才能夠有效處理肌電訊號

In [9]:
package = 50 #單位：毫秒                          #要不要試著調整看看package的大小呀~~
pointer = 0
loop = np.size(data3)-package
IEMG_EMG = np.zeros(loop)


for i in data3:
  raw = data3[pointer:pointer+package-1]
  IEMG_EMG[pointer] = np.mean(np.abs(raw))
  pointer+=1
  if(pointer == loop):
    break


In [ ]:
#稍微設定一下畫布資訊！
start_time = 2 #單位：秒
end_time = 8 #單位：秒
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=IEMG_EMG[start_time*1000:end_time*1000], #取用其中一段肌電訊號
    mode='lines',
    name='Original Plot',
)
)
fig.update_layout(
    title="其中一段的肌電訊號",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
fig.show()

In [ ]:
#將尚不平滑的IEMG_EMG訊號，透過MA方法優化吧！
scale = 200 #單位：毫秒
MA_IEMG_EMG = moving_average(IEMG_EMG,scale)


#稍微設定一下畫布資訊！
start_time = 2 #單位：秒
end_time = 8 #單位：秒

fig = go.Figure()
fig.add_trace(go.Scatter(
    y=MA_IEMG_EMG[start_time*1000:end_time*1000], #取用其中一段肌電訊號
    mode='lines',
    name='Original Plot',
)
)
fig.update_layout(
    title="其中一段的肌電訊號",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
fig.show()

#小結


---


1.   RMS和IEMG都能夠有效處理肌電訊號
2.   MA能夠平滑化稜稜角角的波形
3.   有沒有可能不使用MA也得到平滑的訊號呢？
